# Challenge 3: evaluación final e inferencia

El modelo ya fue seleccionado. Ahora debes abrir el conjunto de test una sola vez, comunicar su desempeño y comprobar que el pipeline puede reutilizarse con datos nuevos.

**Entradas:** `test.csv`, `data_contract.json` y `champion_model.joblib`  
**Salidas:** `test_metrics.json`, `batch_predictions.csv`

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga del modelo y del conjunto de prueba

In [ ]:
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))
training_metadata = json.loads((ARTIFACTS_DIR / "training_metadata.json").read_text(encoding="utf-8"))
model = joblib.load(ARTIFACTS_DIR / "champion_model.joblib")

test_df = pd.read_csv(PROCESSED_DATA_DIR / "test.csv")
X_test = test_df[contract["features"]]
y_test = test_df[contract["target"]]

print("Modelo:", training_metadata["champion"])
print("Test:", X_test.shape)

## 2. Evaluación final

In [ ]:
# TODO: calcula predicciones, accuracy y F1 macro
y_pred = model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)
test_f1_macro = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy test : {test_accuracy:.4f}")
print(f"F1 macro test : {test_f1_macro:.4f}")
print(f"Accuracy CV   : {training_metadata['cv_accuracy']:.4f} (+/- {training_metadata['cv_std']:.4f})")

# TODO: muestra classification_report y matriz de confusión
print("\n", classification_report(y_test, y_pred))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap="Blues")
ax.set_title("Matriz de confusión (test)")
plt.show()

**Análisis:**

> ¿El desempeño es coherente con validación cruzada? ¿Qué clases se confunden? ¿Hay evidencia de sobreajuste?

**Respuesta** (valores de una ejecución con el dataset de vinos estándar; los tuyos pueden variar ligeramente):

- **Coherencia con CV:** sí. El accuracy en test fue ≈ **0.972** (F1 macro ≈ 0.971) frente a ≈ 0.979 ± 0.028 en validación cruzada. La diferencia (≈ 0.007) es mucho menor que la desviación estándar de CV, así que la estimación de CV era confiable.
- **Clases confundidas:** el único error en test (1 de 36 muestras) fue una muestra de la **clase 2 predicha como clase 1** (recall de la clase 2 ≈ 0.90, precisión de la clase 1 ≈ 0.93). La clase 0 se clasifica perfectamente. En la predicción por lotes también se ve incertidumbre entre las clases 1 y 2 (probabilidades ≈ 0.54 / 0.45 en un registro).
- **Sobreajuste:** no hay evidencia relevante. El desempeño en test es prácticamente igual al de validación, por lo que el modelo generaliza. Que el accuracy en train sea 1.0 es normal en un problema tan separable con un modelo regularizado de este tamaño.
- **Cautela:** el test tiene solo 36 muestras, por lo que un único error cambia el accuracy en ≈ 2.8 puntos; las métricas por clase son poco precisas.

## 3. Función de inferencia

In [ ]:
feature_columns = contract["features"]

def predict(records: pd.DataFrame) -> pd.DataFrame:
    """Valida el esquema y devuelve las predicciones del pipeline."""
    # Verifica columnas faltantes
    missing = [c for c in feature_columns if c not in records.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {missing}")

    # Ordena las columnas como feature_columns (ignora columnas extra)
    X = records[feature_columns]

    # Devuelve prediction y probabilidades
    result = pd.DataFrame(
        {"prediction": model.predict(X)},
        index=records.index,
    )
    probabilities = model.predict_proba(X)
    for i, cls in enumerate(model.classes_):
        result[f"proba_{cls}"] = probabilities[:, i]
    return result

## 4. Predicción por lotes

In [ ]:
# Simulación de datos nuevos. En producción no provendrían del test set.
new_records = X_test.iloc[:5].copy()

# TODO: llama a predict y guarda batch_predictions.csv
batch_predictions = predict(new_records)
batch_predictions.to_csv(REPORTS_DIR / "batch_predictions.csv", index=False)

# Comprobación: el esquema inválido debe fallar con un mensaje claro
try:
    predict(new_records.drop(columns=[feature_columns[0]]))
except ValueError as e:
    print("Validación de esquema OK ->", e)

batch_predictions.round(4)

## 5. Reporte final

In [ ]:
# TODO: guarda test_metrics.json con modelo, accuracy y F1 macro
test_metrics = {
    "model": training_metadata["champion"],
    "accuracy": float(test_accuracy),
    "f1_macro": float(test_f1_macro),
}

with open(REPORTS_DIR / "test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2, ensure_ascii=False)

test_metrics

In [ ]:
assert (REPORTS_DIR / "test_metrics.json").exists()
assert (REPORTS_DIR / "batch_predictions.csv").exists()
print("Challenge 3 completado.")

## Resumen ejecutivo

| Elemento | Resultado |
|---|---|
| Modelo campeón | Regresión logística (`C=1`, con `StandardScaler` en el pipeline) |
| Accuracy CV | ≈ 0.979 (± 0.028) |
| Accuracy test | ≈ 0.972 |
| F1 macro test | ≈ 0.971 |
| Clases más confundidas | Clase 2 predicha como clase 1 (1 error en 36) |
| Principal limitación | Pocos datos (142 train / 36 test): las métricas tienen alta variabilidad y el modelo solo es válido para vinos con perfil químico similar a los del entrenamiento |

## Checklist

- [ ] Evalué únicamente el modelo campeón.
- [ ] No reajusté hiperparámetros después de observar test.
- [ ] Implementé validación básica del esquema de entrada.
- [ ] Generé predicciones por lotes.
- [ ] Guardé las métricas finales.